# 03 — Random forest: diagnosis and tuning (Manel)

Development set only (5-fold CV); the test set is not used.

**Scripts run, this notebook explains.** Every number here is produced by
`tuning/random_forest.py` and saved in `tuning/results/random_forest/`; this
notebook only reads those files and comments on them.

| Step | Command (from the project root) | Output |
|---|---|---|
| 1-2. Baseline + diagnosis | `python -m tuning.random_forest baseline` | `baseline.csv` |
| 3. Validation curves | `python -m tuning.random_forest curves` | `curves.csv`, `curves/*.png` |
| 4. Tuning v2 | `python -m tuning.tune --version v2 --model rf` | `tuning/results/v2/best_params.json` |

Tuning v1 = Soumaya's first Optuna run: `tuning/results/v1/best_params.json` (git tag `baseline-v1`).

Hyperparameters studied: `max_depth`, `min_samples_leaf`, `min_samples_split`, `max_features`, `max_samples`, `criterion`, `n_estimators`, `class_weight`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "tuning" / "results" / "random_forest"

## 1. Baseline

**How a random forest "minimises the loss".** Unlike logistic regression or
LightGBM, a random forest does not minimise one global log-loss. Each tree
chooses, at every node, the split that reduces **impurity** the most (Gini by
default: how mixed the two classes are in a node). A fully grown tree keeps
splitting until its leaves are pure, so it memorises the training data. The
forest averages many trees, each trained on a bootstrap sample and a random
subset of features, which reduces that variance.

So most RF hyperparameters control **how far each tree may grow**
(`max_depth`, `min_samples_leaf`, `min_samples_split`) or **how different the
trees are** (`max_features`, `max_samples`).

Five starting points, from the floor to the tuned v1 model (5-fold CV):

In [2]:
baseline = pd.read_csv(RESULTS / "baseline.csv")
baseline[["experiment", "train PR-AUC", "val PR-AUC", "val PR-AUC std", "gap PR-AUC",
          "val ROC-AUC", "val Brier"]].round(3)

,experiment,train PR-AUC,val PR-AUC,val PR-AUC std,gap PR-AUC,val ROC-AUC,val Brier
0,Dummy (predicts the base rate),0.239,0.239,0.000,-0.000,0.500,0.182
1,"One decision tree, fully grown",0.977,0.444,0.011,0.532,0.734,0.196
2,"RF sklearn defaults (100 trees, fully grown)",0.972,0.639,0.009,0.333,0.857,0.131
3,"RF baseline (300 trees, min_samples_leaf 5)",0.803,0.711,0.012,0.092,0.890,0.111
4,RF tuned v1,0.749,0.715,0.013,0.035,0.891,0.136


PR-AUC: higher is better; 0.239 is the floor (the share of high earners).
Brier: quality of the probabilities, lower is better.

## 2. Diagnosis: train vs validation (overfitting or underfitting?)

1. **One fully grown tree memorises:** 0.977 on the data it learned from,
   0.444 on new data (gap 0.53). High variance.
2. **A forest averages part of that noise away, not all of it:** sklearn
   defaults reach 0.639, but the trees are still fully grown and the gap is
   still 0.33.
3. **Bigger leaves fix the overfitting:** at least 5 people per leaf
   (`min_samples_leaf=5`) shrinks the gap to 0.09 and lifts validation to
   0.711. The parameters that limit tree growth are the ones that matter most.
4. **Tuning v1 adds nothing measurable:** 0.711 → 0.715, while the fold-to-fold
   noise is ± 0.013. Its gap (0.035) is small: overfitting is no longer the problem.
5. **Tuning v1 made the probabilities worse:** Brier 0.111 → 0.136, because of
   `class_weight="balanced_subsample"`, which pushes every probability up. The
   ranking is unchanged (ROC-AUC 0.890 vs 0.891), but the probabilities can no
   longer be read as real chances, which matters for calibration and fairness.
6. **The ceiling is the data, not the settings:** LR 0.708, RF 0.715 and
   LightGBM 0.716 all stop around 0.71 PR-AUC with 6 inputs. Expect tuning to
   move results by about ± 0.01.

**Questions for the validation curves:** where does the gap close without
validation dropping (`min_samples_leaf`, `max_depth`, `min_samples_split`)?
Does decorrelating the trees help (`max_features`, `max_samples`)? Does class
weighting cost calibration when tested alone (`class_weight`, Brier)?

## 3. Validation curves on the key hyperparameters

One hyperparameter changes, the others stay at the sklearn defaults with 200
trees. Run `python -m tuning.random_forest curves` (all curves) or
`python -m tuning.random_forest curves --param max_depth` (one curve), then
re-run the cell below.

In [3]:
path = RESULTS / "curves.csv"
if path.exists():
    curves = pd.read_csv(path)
    for param, table in curves.groupby("param", sort=False):
        print(param)
        display(table[["value", "train PR-AUC", "val PR-AUC", "val PR-AUC std",
                       "gap PR-AUC", "val Brier"]].round(3))
        display(Image(str(RESULTS / "curves" / f"{param}.png")))
else:
    print("No curves yet: run  python -m tuning.random_forest curves")

No curves yet: run  python -m tuning.random_forest curves


## 4. Tuning v2

## 5. Findings